In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# set gpu to 1
torch.cuda.set_device(1)

In [ ]:
base_model_id = "mistralai/Mistral-7B-v0.1"
bnb_config = BitsAndBytesConfig(
    # load_in_8bit=True,
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,  # Mistral, same as before
    quantization_config=bnb_config,  # Same quantization config as before
    device_map="auto",
    trust_remote_code=True,
    use_auth_token=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

In [ ]:
from peft import PeftModel
ft_model = PeftModel.from_pretrained(base_model, "mistral-nordavind/checkpoint-250")

In [ ]:
ft_model.eval();

In [ ]:
SYS_PROMPT = "En samtale mellom en person og en AI-assistent kalt Nordavind"
USR_PROMPT = "Skriv kode i c++, java, javascript og ruby som itererer over en liste med tall og skriver ut hver verdi ganget med to"
USR_PROMPT = "Sammenlign de største politiske partiene i Norge."
USR_PROMPT = "Oppsummer følgende: Et boktrykkeri er et kommersielt foretagende for trykking av bøker og andre typer dokumenter for mangfoldiggjøring og distribusjon ved hjelp av høytrykkteknologi. Boktrykkeriet som institusjon er primært knyttet til en periode fra boktrykkerkunstens oppfinnelse på midten av 1400-tallet til siste halvdel av 1900-tallet. Boktrykkeriene var den vestlige verdens første, og lenge eneste, system for massedistribusjon av informasjon."

Q = f"{SYS_PROMPT} [INST] {USR_PROMPT} [/INST]"

MAX_NEW = 512
model_input = tokenizer(Q, return_tensors="pt").to("cuda:1")
pen = 1.1
with torch.no_grad():
    gen = tokenizer.decode(
        ft_model.generate(
            **model_input,
            max_new_tokens=MAX_NEW,
            repetition_penalty=pen,
        )[0],
        skip_special_tokens=True,
    )
    gen = gen.split("[/INST]")[-1].strip()
    print("PROMPT:", USR_PROMPT)
    print()
    print("OUTPUT:", gen)
    # print(tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)[0], skip_special_tokens=True))
